# Datathon Passos Mágicos - Análise Exploratória e Storytelling

Este notebook responde às 11 perguntas do desafio com base na base consolidada analítica e, quando necessário, na base reduzida para modelagem.

## Objetivos
- Entender a evolução dos indicadores pedagógicos, psicossociais e de engajamento;
- Identificar padrões de risco de defasagem;
- Avaliar a efetividade do programa ao longo dos anos e fases;
- Preparar insumos para o modelo preditivo e para o storytelling da apresentação final.

## Bases utilizadas
- Base analítica consolidada: `base_PEDE_consolidada_analitica.parquet`
- Base reduzida para ML: `base_processada_reduzida_ML.parquet`

In [ ]:
# 2) Imports e configuraçãoimport warnings
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

sns.set_theme(style="whitegrid")

In [ ]:
# 3) Carga das bases
df = pd.read_parquet("../data/processed/base_PEDE_consolidada_analitica.parquet")
df_ml = pd.read_parquet("../data/processed/base_processada_reduzida_ML.parquet")

print("Base analítica:", df.shape)
print("Base ML:", df_ml.shape)

display(df.head())
display(df_ml.head())


In [ ]:
# 4) Padronizações auxiliares
# cria coluna inde única na base analítica
df["inde"] = np.nan

if "inde_2022" in df.columns:
    df.loc[df["ano_pede"] == 2022, "inde"] = df["inde_2022"]
if "inde_2023" in df.columns:
    df.loc[df["ano_pede"] == 2023, "inde"] = df["inde_2023"]
if "inde_2024" in df.columns:
    df.loc[df["ano_pede"] == 2024, "inde"] = df["inde_2024"]

# garantir numéricos
cols_numericas = [
    "idade", "ano_ingresso", "inde", "iaa", "ieg", "ips", "ipp",
    "ida", "mat", "por", "ing", "ipv", "ian", "defasagem"
]

for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# categorias úteis
for col in ["fase", "turma", "genero", "instituicao_ensino", "pedra_2022", "pedra_2023", "pedra_2024"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

df.info()

In [ ]:
# 5) Funções auxiliares
def plot_bar(data, x, y, title, figsize=(10,5), rotation=0):
    plt.figure(figsize=figsize)
    sns.barplot(data=data, x=x, y=y)
    plt.title(title)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()


def plot_line(data, x, y, hue=None, title="", figsize=(10,5), rotation=0, marker="o"):
    plt.figure(figsize=figsize)
    sns.lineplot(data=data, x=x, y=y, hue=hue, marker=marker)
    plt.title(title)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()


def plot_heatmap_corr(df_corr, title="Mapa de Correlação"):
    plt.figure(figsize=(10, 7))
    sns.heatmap(df_corr, annot=True, cmap="Blues", fmt=".2f")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def categorizar_defasagem(valor):
    if pd.isna(valor):
        return "Sem informação"
    if valor <= 0:
        return "Sem defasagem"
    elif valor == 1:
        return "Defasagem leve"
    elif valor == 2:
        return "Defasagem moderada"
    else:
        return "Defasagem severa"

### 5. Aspectos psicossociais (IPS)

In [ ]:
# 23) IPS e desempenho acadêmico/engajamento
corr_cols = [c for c in ["ips", "ida", "ieg", "ipv"] if c in df.columns]
corr_ips = df[corr_cols].corr()
display(corr_ips)

plot_heatmap_corr(corr_ips, "Correlação entre IPS, IDA, IEG e IPV")

### Interpretação dos resultados

A matriz de correlação indica que o IPS apresenta correlação praticamente nula com IDA, IEG e IPV, sugerindo que, de forma isolada, o indicador psicossocial não explica diretamente a variação no desempenho acadêmico, no engajamento ou no ponto de virada dos alunos.

Por outro lado, observa-se correlação moderada e positiva entre IEG e IDA (0,54), entre IDA e IPV (0,56) e entre IEG e IPV (0,56). Esses resultados indicam que alunos mais engajados tendem a apresentar melhor desempenho acadêmico, e que tanto o desempenho quanto o engajamento estão associados a um melhor ponto de virada.

Em termos práticos, os achados sugerem que estratégias voltadas ao aumento do engajamento podem gerar efeitos positivos tanto sobre o desempenho escolar quanto sobre a evolução global do aluno no programa.

### Insight de negócio para a Passos Mágicos

O principal insight desse gráfico é que engajamento e desempenho caminham juntos e ambos estão fortemente associados ao ponto de virada. Isso sugere que o IEG pode ser tratado como um indicador precoce de acompanhamento, ajudando a identificar alunos que podem precisar de intervenção antes que ocorra piora no desempenho acadêmico.

### Cuidado metodológico

É importante destacar que correlação não implica causalidade. O gráfico mostra associações lineares entre os indicadores, mas não permite afirmar, isoladamente, que uma variável causa a outra. Portanto, a leitura correta é que os indicadores estão relacionados e variam conjuntamente.

In [ ]:
# 24) Faixas de IPS
df["faixa_ips"] = pd.qcut(df["ips"], q=4, labels=["Baixo", "Médio-baixo", "Médio-alto", "Alto"])

ips_resumo = (
    df.groupby("faixa_ips", observed=False)[["ida", "ieg", "ipv"]]
      .mean()
      .reset_index()
)

display(ips_resumo)

### Interpretação da análise por faixas de IPS

Para aprofundar a análise do indicador psicossocial (IPS), os alunos foram divididos em quatro grupos (quartis): Baixo, Médio-baixo, Médio-alto e Alto. Em seguida, foram calculadas as médias de desempenho acadêmico (IDA), engajamento (IEG) e ponto de virada (IPV) para cada faixa.

Observa-se que os alunos com IPS mais alto apresentam, em média, melhores resultados em todos os indicadores analisados.

- O grupo com IPS alto apresenta os maiores valores médios de IDA, IEG e IPV.
- O grupo com IPS médio-baixo apresenta os menores valores médios de IEG e IPV.
- Há uma tendência geral de aumento dos indicadores conforme o IPS aumenta, embora a diferença não seja extremamente grande.

Esse resultado sugere que aspectos psicossociais mais positivos estão associados a melhores níveis de engajamento, desempenho acadêmico e evolução no ponto de virada.

### Insight

Embora a correlação linear entre IPS e os demais indicadores tenha sido baixa, a análise por faixas mostra um padrão mais claro: alunos com melhor avaliação psicossocial tendem a apresentar melhores resultados gerais.

Isso indica que o IPS pode não atuar de forma linear, mas sim como um fator de suporte que favorece melhores resultados quando combinado com outros indicadores, como engajamento e desempenho acadêmico.

### Implicação para o programa

O acompanhamento do IPS pode ser útil para identificar alunos que precisam de apoio socioemocional, mesmo antes de aparecer queda nas notas ou no engajamento. Intervenções psicossociais preventivas podem contribuir para melhorar o desempenho global do aluno ao longo do programa.

In [ ]:
# 25) Indício longitudinal com RA
if "ra" in df.columns:
    df = df.sort_values(["ra", "ano_pede"]).copy()
    df["ida_ano_seguinte"] = df.groupby("ra")["ida"].shift(-1)
    df["ieg_ano_seguinte"] = df.groupby("ra")["ieg"].shift(-1)

    df["delta_ida_futuro"] = df["ida_ano_seguinte"] - df["ida"]
    df["delta_ieg_futuro"] = df["ieg_ano_seguinte"] - df["ieg"]

    ips_future = df[["ips", "delta_ida_futuro", "delta_ieg_futuro"]].corr()
    display(ips_future)

### Leitura analítica
- Se IPS alto estiver associado a melhor IDA, IEG e IPV, os aspectos psicossociais parecem funcionar como fator protetivo.
- A análise com ano seguinte é uma aproximação útil para avaliar se IPS antecede melhora ou queda futura.